In [14]:
import json
from collections import Counter
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

RAW = Path("../data/raw")
FPL = RAW / "fpl"

def read_match(season: str) -> pd.DataFrame:
    """Read one raw season CSV, tolerating encoding drift in older files."""
    path = RAW / f"E0_{season}.csv"
    for encoding in ("utf-8-sig","latin-1"):
        try:
            return pd.read_csv(path, encoding=encoding)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"could not decode {path}")

def read_fpl(name: str):
    return json.loads((FPL / f"{name}.json").read_text())

SEASONS = sorted(p.stem.split("_")[1] for p in RAW.glob("E0_*.csv"))
SEASONS

['1516',
 '1617',
 '1718',
 '1819',
 '1920',
 '2021',
 '2122',
 '2223',
 '2324',
 '2425',
 '2526',
 '2627']

### 1 · What did I actually download?

In [15]:
manifest = json.loads((RAW / "match_manifest.json").read_text())
pd.DataFrame(manifest).T[["rows", "bytes", "fetched_at"]]

,rows,bytes,fetched_at
1516,380,100656,2026-09-21T17:30:02+00:00
1617,380,99990,2026-09-21T17:30:04+00:00
1718,380,100734,2026-09-21T17:30:06+00:00
1819,380,96144,2026-09-21T17:30:08+00:00
1920,380,175475,2026-09-21T17:30:10+00:00
2021,380,175815,2026-09-21T17:30:11+00:00
2122,380,175363,2026-09-21T17:30:13+00:00
2223,380,176180,2026-09-21T17:30:15+00:00
2324,380,172196,2026-09-21T17:30:17+00:00
2425,380,197110,2026-09-21T17:30:19+00:00


### 2 · Which columns exist in every season file?


In [16]:
presence = {}
for season in SEASONS:
    for col in read_match(season).columns:
        presence.setdefault(col, []).append(season)

stable = sorted(c for c, s in presence.items() if len(s) == len(SEASONS))
unstable = {c: sorted(set(SEASONS) - set(s))
            for c, s in presence.items() if len(s) < len(SEASONS)}

print(f"stable ({len(stable)}):\n  ", ", ".join(stable))
print(f"\nunstable ({len(unstable)}) — first 10:")
for col, missing in list(unstable.items())[:10]:
    print(f"   {col:12} missing from {missing}")

stable (29):
   AC, AF, AR, AS, AST, AY, AwayTeam, B365A, B365D, B365H, BWA, BWD, BWH, Date, Div, FTAG, FTHG, FTR, HC, HF, HR, HS, HST, HTAG, HTHG, HTR, HY, HomeTeam, Referee

unstable (165) — first 10:
   IWH          missing from ['2425', '2526', '2627']
   IWD          missing from ['2425', '2526', '2627']
   IWA          missing from ['2425', '2526', '2627']
   LBH          missing from ['1819', '1920', '2021', '2122', '2223', '2324', '2425', '2627']
   LBD          missing from ['1819', '1920', '2021', '2122', '2223', '2324', '2425', '2627']
   LBA          missing from ['1819', '1920', '2021', '2122', '2223', '2324', '2425', '2627']
   PSH          missing from ['2627']
   PSD          missing from ['2627']
   PSA          missing from ['2627']
   WHH          missing from ['2526', '2627']


### 3 · What do the dates actually look like?


In [17]:
for season in SEASONS[:3] + SEASONS[-2:]:
    sample = read_match(season)["Date"].dropna()
    lengths = Counter(sample.str.len())
    print(f"{season}: {sample.iloc[0]!r}  length counts {dict(lengths)}")

1516: '08/08/2015'  length counts {10: 380}
1617: '13/08/16'  length counts {8: 380}
1718: '11/08/2017'  length counts {10: 380}
2526: '15/08/2025'  length counts {10: 380}
2627: '21/08/2026'  length counts {10: 40}


### 4 · Every distinct team name, in both sources


In [18]:
match_names = set()
for season in SEASONS:
    df = read_match(season)
    match_names |= set(df["HomeTeam"].dropna()) | set(df["AwayTeam"].dropna())

boot = read_fpl("bootstrap")
fpl_names = {t["name"] for t in boot["teams"]}

print(f"match sources: {len(match_names)} distinct names")
print(f"FPL:           {len(fpl_names)} distinct names")
print("\nin FPL but spelled differently in the match files:")
print(sorted(fpl_names - match_names))
print("\nin the match files but not in FPL (relegated / historical):")
print(sorted(match_names - fpl_names))

match sources: 35 distinct names
FPL:           20 distinct names

in FPL but spelled differently in the match files:
['Coventry City', 'Hull City', 'Ipswich Town', 'Man Utd', 'Spurs']

in the match files but not in FPL (relegated / historical):
['Burnley', 'Cardiff', 'Coventry', 'Huddersfield', 'Hull', 'Ipswich', 'Leicester', 'Luton', 'Man United', 'Middlesbrough', 'Norwich', 'Sheffield United', 'Southampton', 'Stoke', 'Swansea', 'Tottenham', 'Watford', 'West Brom', 'West Ham', 'Wolves']


### 5 · Is now_cost really in tenths?


In [19]:
players = pd.DataFrame(boot["elements"])
expensive = players.nlargest(5, "now_cost")[
    ["web_name", "now_cost", "total_points", "minutes"]]
expensive["price_if_tenths"] = expensive["now_cost"] / 10
expensive

,web_name,now_cost,total_points,minutes,price_if_tenths
491,Haaland,156,39,450,15.6
517,B.Fernandes,120,31,450,12.0
176,Palmer,97,28,442,9.7
11,Saka,95,32,416,9.5
459,Isak,91,33,413,9.1


### 6 · How many players have never played?


In [20]:
positions = {t["id"]: t["singular_name_short"] for t in boot["element_types"]}
players["position"] = players["element_type"].map(positions)

summary = players.groupby("position").agg(
    total=("id", "size"),
    zero_minutes=("minutes", lambda s: (s == 0).sum()),
    median_minutes=("minutes", "median"),
)
summary["pct_unused"] = (100 * summary.zero_minutes / summary.total).round(0)
summary.sort_values("pct_unused", ascending=False)

,total,zero_minutes,median_minutes,pct_unused
position,,,,
GKP,73,49,0.0,67.0
FWD,79,29,31.0,37.0
MID,298,99,50.0,33.0
DEF,217,69,113.0,32.0


### 7 · Can a player appear twice in one gameweek?


In [21]:
sample_id = int(players.nlargest(1, "total_points")["id"].iloc[0])
history = pd.DataFrame(read_fpl(f"player_{sample_id}")["history"])

per_round = history.groupby("round").size()
print("rows per gameweek:", dict(per_round))
print("\nany gameweek with more than one fixture:",
      per_round[per_round > 1].to_dict() or "none yet this season")
history[["round", "fixture", "opponent_team", "minutes", "total_points"]].head(8)

rows per gameweek: {1: np.int64(1), 2: np.int64(1), 3: np.int64(1), 4: np.int64(1), 5: np.int64(1)}

any gameweek with more than one fixture: none yet this season


,round,fixture,opponent_team,minutes,total_points
0,1,7,2,90,2
1,2,16,6,90,13
2,3,23,13,90,1
3,4,38,7,90,17
4,5,42,1,90,14


### 8 · What is actually in a player history payload?


In [22]:
payload = read_fpl(f"player_{sample_id}")
print("top-level keys:", list(payload))
print(f"\nhistory fields ({len(history.columns)}):")
print("  ", ", ".join(sorted(history.columns)))

past = pd.DataFrame(payload.get("history_past", []))
print(f"\nhistory_past rows: {len(past)}")
if not past.empty:
    print(past[["season_name", "total_points", "start_cost", "end_cost"]].tail())

top-level keys: ['fixtures', 'history', 'history_past']

history fields (41):
   assists, bonus, bps, clean_sheets, clearances_blocks_interceptions, creativity, defensive_contribution, element, expected_assists, expected_goal_involvements, expected_goals, expected_goals_conceded, fixture, goals_conceded, goals_scored, ict_index, influence, kickoff_time, minutes, modified, opponent_team, own_goals, penalties_missed, penalties_saved, recoveries, red_cards, round, saves, selected, starts, tackles, team_a_score, team_h_score, threat, total_points, transfers_balance, transfers_in, transfers_out, value, was_home, yellow_cards

history_past rows: 9
  season_name  total_points  start_cost  end_cost
4     2021/22            88          60        56
5     2022/23           159          55        54
6     2023/24           153          65        61
7     2024/25             0          65        65
8     2025/26            78          55        56


### 9 · Do the two sources agree on fixtures?


In [23]:
fixtures = pd.DataFrame(read_fpl("fixtures"))
fixtures["kickoff"] = pd.to_datetime(fixtures["kickoff_time"], utc=True)
played = fixtures[fixtures["finished"]]

live = read_match(SEASONS[-1])
print(f"FPL finished fixtures: {len(played)}")
print(f"match file rows:       {len(live.dropna(subset=['FTR']))}")
print("\nFPL date range:", played["kickoff"].min(), "to", played["kickoff"].max())

FPL finished fixtures: 50
match file rows:       40

FPL date range: 2026-08-21 19:00:00+00:00 to 2026-09-20 15:30:00+00:00
